In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')
adata.obsm = tmp_adata.obsm.copy()

In [ ]:
clusters_path = f'{projdir}/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pl.umap(adata, color=['coarse', 'time_num'])

In [ ]:
genes = ['IFI44L', 'XAF1', 'PRRX1',
         'PARP14', 'EPSTI1', 'ADAMTS12']
indices = adata.var.index.get_indexer(genes)
for g,idx in zip(genes, indices):
    adata.obs[g] = np.ravel(adata[:,idx].X.toarray())

In [ ]:
tmp_adata = adata.copy()
tmp_adata.var = tmp_adata.var.reset_index(drop=True)
sc.pl.umap(tmp_adata, color=genes, ncols=3)

In [ ]:
umap_df = pd.DataFrame(index=adata.obs.index)
umap_df['umap0'] = adata.obsm['X_umap'][:,0]
umap_df['umap1'] = adata.obsm['X_umap'][:,1]

select = ['time_num', 'coarse'] + genes
for s in select:
    umap_df[s] = adata.obs[s]

In [ ]:
umap_df.head()

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']
mask = np.array([x in whitelist for x in adata.obs['coarse']])
fail_adata = adata[mask]
fail_adata

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
umap_df.to_csv(f'{outdir}/multiome_rna_immune_fail.csv', sep='\t')

### PRRX1 - donor-specific?

In [ ]:
adata.obs.head()

In [ ]:
sns.boxplot(data=fail_adata.obs, x='coarse', y='PRRX1', hue='assignment')

In [ ]:
!date